In [1]:
import bw2data, bw2io, bw2calc
from bw_timex import TimexLCA
from bw_temporalis import TemporalDistribution, easy_timedelta_distribution
import numpy as np
from datetime import datetime
import os
import re
import pandas as pd
import numpy as np
import pickle

In [2]:
import sys
sys.path.append('../utils/') 
from elec_builder import *

In [3]:
# activate the bw project
bw2data.projects.set_current("ei311")
#for db in bw2data.databases:
#    print(db, len(bw2data.Database(db)))

In [5]:
### just a new ONE FG act  in the foreground database 

hydro_qc = ["CA-ON"]

xx = build_dynamic_electricity_all(
    locations = hydro_qc , 
    pathways = [  "SSP5-H" ],
    years = [2050], 
    elec_act = 'electricity production, hydro, reservoir, non-alpine region', #'electricity production, hydro, run-of-river',
    ref_name = 'market for electricity, hydro, high voltage',
    fg_db_name="elec_foreground",
    flush_fg_db = True
)

pathways should match the premise database names, e.g., SSP1-VLLO, SSP2-M, SSP5-H 
 also IAM locations for premise database can be found through calling: 
 from premise import geomap -> premise.geomap.Geomap('image').iam_regions 
Flushing existing foreground DB: 'elec_foreground'
Created fresh foreground DB: 'elec_foreground'


In [6]:
one_hydro = bw2data.Database("elec_foreground")
list(one_hydro)

['market for electricity, hydro, high voltage, CA-ON, SSP5-H, 2050' (kWh, CA-ON, None)]

In [9]:
rows = []   # collect results for all acts
act_list = list(one_hydro)   
for act in act_list: 
    print(act)

    name = act.get("name")
    name_parts = [p.strip() for p in name.split(",")]
    # run static LCI + premise_GWP vs. pGWP100 first: 
    pgwp_fixedco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), method_suffix = "pGWP100 - fixed-AGWPCO2")
    pgwp_dpco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), method_suffix = "pGWP100 - dp-AGWPCO2")
    gwp =  ('ecoinvent-3.11', 'IPCC 2021 (incl. biogenic CO2)', 'climate change: total (incl. biogenic CO2, incl. SLCFs)', 'global warming potential (GWP100)')   

    lca_gwp = bw2calc.lca.LCA({act: 1}, method=gwp)
    lca_gwp.lci(); lca_gwp.lcia()
    score_gwp = float(lca_gwp.score)

    lca_fixed = bw2calc.lca.LCA({act: 1}, method=pgwp_fixedco2)
    lca_fixed.lci(); lca_fixed.lcia()
    score_fixedco2 = float(lca_fixed.score)

    lca_dp = bw2calc.lca.LCA({act: 1}, method=pgwp_dpco2)
    lca_dp.lci(); lca_dp.lcia()
    score_dpco2 = float(lca_dp.score)

    print(score_gwp, score_fixedco2, score_dpco2) 

    # ---- store results for this activity ----
    rows.append({
        "Activity": name,                      # index value later
        "gwp100": score_gwp,
        "pGWP100_fixedCO2": score_fixedco2,
        "pGWP100_dpCO2": score_dpco2,
    })


'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2050' (kWh, CA-QC, None)
0.015582490374858192 0.014036297672589822 0.01552817913234485


In [7]:
database_dates = {
    #'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.strptime("2030", "%Y"),
    #'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    #'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    #'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    #'ei_cutoff_3.11_image_SSP2-M_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    #'ei_cutoff_3.11_image_SSP2-M_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    #'ei_cutoff_3.11_image_SSP5-H_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    #'ei_cutoff_3.11_image_SSP5-H_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2050 2025-11-22': datetime.strptime("2050", "%Y"),
    
    "elec_foreground": "dynamic", # flag databases that should be temporally distributed with "dynamic"
}

In [9]:
hydro_db_2050 = [
    act for act in one_hydro 
    if  "2050" in str(act.get('name', '')) and  "CA-" in str(act.get('name', '')) and  "SSP5" in str(act.get('name', ''))   #or  "2030" in str(act.get('name', '') )
]

hydro_db_2050

['market for electricity, hydro, high voltage, CA-ON, SSP5-H, 2050' (kWh, CA-ON, None)]

In [10]:
dp_results = {}

for act in list(hydro_db_2050): 
    print(act)
 
    #### dynamic LCI:: 
    assign_td_from_foreground_db( 
        select_act = act,
        elec_td_year=10,
        resolution="Y",
        kind="uniform",
        fg_db_name="elec_foreground", #"elec_hydro_reservoir_foreground", 
        verbose = True
     )

    tlca = run_dp_timex_lca(foreground_act = act ,   
                    pathway = None,
                    year = None,
                    method = None,
                    database_dates = None, # database_dates, #None not working, has to incl. all 9 background DB ... 
                    temporal_grouping="year", 
                    method_prefix = "Climate Change prospective GWP100",
                    method_suffix = "pGWP100 - fixed-AGWPCO2", 
                    fg_db_name = 'elec_foreground'
     )

    tlca.lci()
    tlca.dynamic_inventory.shape

    # dyn-foreground LCI + static background LCI + pGWP100
    lca_0 = tlca.base_score 
    print(lca_0)
    # dpLCI (dyn-foreground LCI + dyn background LCI) +  pGWP100  .static_score is the full dpLCI but with non-dyn LCIA
    tlca.static_lcia()
    lca_1 = tlca.static_score
    print(lca_1)

    act_name = act.get("name")
    dp_results[act_name] = {
        "dyn FG LCI static BG p-LCI, pGWP100-fixedCO2": float(lca_0),   # static BG LCI + dyn FG LCI
        "full dp-LCI, pGWP100-fixedCO2": float(lca_1),      # dyn BG LCI + dyn FG LCI
    }

    #### now save all dyLCI flows to pandas 
    df = tlca.dynamic_inventory_df

    ##### important to convert flow and act as str to excel 
    df["flow"] = df["flow"].astype(str)
    df["activity"] = df["activity"].astype(str)

    ### export df to dp-LCI_output folder, using the act name as the excel name 
    out_dir = "dp-LCI_output/hydro_dpLCI_v2_reservoir"
    os.makedirs(out_dir, exist_ok=True)    
    raw_name = act["name"]
    safe_name = re.sub(r"[^A-Za-z0-9_\-()]+", "_", raw_name)   # replace spaces/special chars
    
    excel_path = os.path.join(out_dir, f"{safe_name}.xlsx")
    
    df.to_excel(excel_path, index=False)    
    print(f"✔ Exported dynamic inventory DF for '{raw_name}' → {excel_path}")



2025-12-18 12:18:21.132 | INFO     | bw_timex.timex_lca:__init__:114 - Initializing TimexLCA object...
2025-12-18 12:18:21.133 | INFO     | bw_timex.timex_lca:__init__:136 - Collecting node infos...


'market for electricity, hydro, high voltage, CA-ON, SSP5-H, 2050' (kWh, CA-ON, None)
TD applied to 'market for electricity, hydro, high voltage, CA-ON, SSP5-H, 2050' → Exchange: 1 kilowatt hour 'electricity production, hydro, reservoir, non-alpine region' (kilowatt hour, CA-ON, None) to 'market for electricity, hydro, high voltage, CA-ON, SSP5-H, 2050' (kWh, CA-ON, None)>
for the activity 'market for electricity, hydro, high voltage, CA-ON, SSP5-H, 2050' (kWh, CA-ON, None),it's under SSP-SSP5-H, year-2050 
 we'll use LCIA ('Climate Change prospective GWP100', 'SSP585', 'MY2050', 'pGWP100 - fixed-AGWPCO2')  with background database.datetime = {'ei_cutoff_3.11_image_SSP5-H_2050 2025-11-22': datetime.datetime(2050, 1, 1, 0, 0), 'elec_foreground': 'dynamic'}



2025-12-18 12:18:25.400 | INFO     | bw_timex.timex_lca:build_timeline:216 - No edge filter function provided. Skipping all edges in background databases.
2025-12-18 12:18:28.366 | INFO     | bw_timex.timex_lca:build_timeline:232 - Calculating base LCA...
2025-12-18 12:18:34.100 | INFO     | bw_timex.timex_lca:build_timeline:242 - Creating activity time mapping...
2025-12-18 12:18:34.343 | INFO     | bw_timex.timeline_builder:__init__:99 - Traversing supply chain graph...
2025-12-18 12:18:34.508 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...
2025-12-18 12:18:34.562 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-18 12:18:34.563 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provi

Starting graph traversal
Calculation count: 1


2025-12-18 12:18:34.939 | INFO     | bw_timex.timex_lca:lci:360 - Expanding matrices...
2025-12-18 12:18:34.957 | INFO     | bw_timex.timex_lca:lci:379 - Calculating dynamic inventory...


0.0451981422104142
0.04519814221041419
✔ Exported dynamic inventory DF for 'market for electricity, hydro, high voltage, CA-ON, SSP5-H, 2050' → dp-LCI_output/hydro_dpLCI_v2_reservoir/market_for_electricity_hydro_high_voltage_CA-ON_SSP5-H_2050.xlsx


so appearantly, with only one foreground activity (a 2050/SSP5 hydro), it can be mapped to ONLY one selected background database_dates (just 2050 SSP background DB), but still the output file has the date column from 2026 to 2036 even if all foreground / background only inlc. 2050 